# Agent Evaluation: Testing Individual Agent Performance

This notebook evaluates each specialized agent in the ScholarSync multi-agent system.
We test agent routing accuracy, response quality, and specific capabilities.

In [1]:
import sys
sys.path.append('..')

import json
import asyncio
from pathlib import Path
from src.agents import AgentRegistry, AgentContext
from src.agents.summarizer import summarizer_agent
from src.agents.methodology_extractor import methodology_extractor_agent
from src.agents.comparator import comparator_agent
from src.agents.gap_finder import gap_finder_agent
from src.agents.citation_analyzer import citation_analyzer_agent
from src.agents.general_qa import general_qa_agent

## 1. Agent Registry Overview

The AgentRegistry manages all specialized agents and handles query routing based on keywords

In [2]:
print("="*60)
print("REGISTERED AGENTS")
print("="*60)

registered_agents = [
    summarizer_agent,
    methodology_extractor_agent,
    comparator_agent,
    gap_finder_agent,
    citation_analyzer_agent,
    general_qa_agent
]

for agent in registered_agents:
    print(f"\n{agent.name}")
    print(f"  Description: {agent.description}")
    print(f"  Trigger Keywords: {', '.join(agent.trigger_keywords)}")

REGISTERED AGENTS

SummarizerAgent
  Description: Generates structured summaries of research papers with key findings, methodology, and contributions
  Trigger Keywords: summarize, summary, summarise, tldr, tl;dr, overview, brief, key points, main points, what is this paper about, explain this paper

MethodologyExtractorAgent
  Description: Extracts research methods, datasets, algorithms, and experimental setup from papers
  Trigger Keywords: methodology, method, methods, approach, technique, algorithm, dataset, experimental setup, how did they, what method, what approach

ComparatorAgent
  Description: Compares multiple papers side-by-side, highlighting similarities, differences, and relative strengths
  Trigger Keywords: compare, comparison, vs, versus, difference, differences, similar, contrast, which is better, how do they differ

GapFinderAgent
  Description: Identifies research gaps, limitations, and future research directions from papers
  Trigger Keywords: gap, gaps, limitation

## 2. Agent Routing Tests

Test if the AgentRegistry correctly routes queries to the appropriate agent

In [3]:
test_queries = [
    ("Can you summarize this paper?", "SummarizerAgent"),
    ("What methodology did they use?", "MethodologyExtractorAgent"),
    ("Compare these two approaches", "ComparatorAgent"),
    ("What are the research gaps?", "GapFinderAgent"),
    ("What papers does this cite?", "CitationAnalyzerAgent"),
    ("What were the main results?", "GeneralQAAgent"),
    ("Give me a tldr", "SummarizerAgent"),
    ("How does approach A differ from B?", "ComparatorAgent"),
]

print("\n" + "="*60)
print("AGENT ROUTING TESTS")
print("="*60)

routing_results = []
for query, expected_agent in test_queries:
    matched_agent = AgentRegistry.find_matching(query)
    agent_name = matched_agent.name if matched_agent else "None"
    is_correct = agent_name == expected_agent
    
    routing_results.append({
        "query": query,
        "expected": expected_agent,
        "matched": agent_name,
        "correct": is_correct
    })
    
    status = "✓" if is_correct else "✗"
    print(f"\n{status} Query: '{query}'")
    print(f"  Expected: {expected_agent}")
    print(f"  Matched:  {agent_name}")

accuracy = sum(r["correct"] for r in routing_results) / len(routing_results) * 100
print(f"\n{'='*60}")
print(f"Routing Accuracy: {accuracy:.1f}% ({sum(r['correct'] for r in routing_results)}/{len(routing_results)})")


AGENT ROUTING TESTS

✓ Query: 'Can you summarize this paper?'
  Expected: SummarizerAgent
  Matched:  SummarizerAgent

✓ Query: 'What methodology did they use?'
  Expected: MethodologyExtractorAgent
  Matched:  MethodologyExtractorAgent

✓ Query: 'Compare these two approaches'
  Expected: ComparatorAgent
  Matched:  ComparatorAgent

✓ Query: 'What are the research gaps?'
  Expected: GapFinderAgent
  Matched:  GapFinderAgent

✓ Query: 'What papers does this cite?'
  Expected: CitationAnalyzerAgent
  Matched:  CitationAnalyzerAgent

✓ Query: 'What were the main results?'
  Expected: GeneralQAAgent
  Matched:  GeneralQAAgent

✓ Query: 'Give me a tldr'
  Expected: SummarizerAgent
  Matched:  SummarizerAgent

✗ Query: 'How does approach A differ from B?'
  Expected: ComparatorAgent
  Matched:  MethodologyExtractorAgent

Routing Accuracy: 87.5% (7/8)


## 3. Individual Agent Evaluation

Test each agent with realistic queries and sample context

In [8]:
# Sample paper context for testing
sample_context = """
(Source: attention_is_all_you_need.pdf, Page 3)
The Transformer model architecture is based entirely on self-attention mechanisms,
dispensing with recurrence and convolutions entirely. The model uses multi-head attention
to allow the model to jointly attend to information from different representation
subspaces at different positions.

(Source: attention_is_all_you_need.pdf, Page 5)
We evaluate our models on machine translation tasks, specifically WMT 2014 English-to-German 
and English-to-French. The model achieves 28.4 BLEU on the English-to-German translation task,
improving over the existing best results by over 2 BLEU points.

(Source: attention_is_all_you_need.pdf, Page 8)
One limitation of our approach is the quadratic complexity of self-attention with respect
to sequence length, making it less suitable for very long sequences. Future work could
explore more efficient attention mechanisms.
"""

# Create a mock agent context with multiple papers for ComparatorAgent
mock_context = AgentContext(
    tenant_id="evaluation",
    papers=["attention_is_all_you_need.pdf", "lstm_networks.pdf"],
    chat_history=[],
    metadata={
        "paper_context": sample_context + """

(Source: lstm_networks.pdf, Page 2)
Long Short-Term Memory (LSTM) networks use recurrent connections with gating mechanisms
to process sequential data while maintaining long-term dependencies. Unlike transformers,
LSTMs process sequences sequentially rather than in parallel.

(Source: lstm_networks.pdf, Page 4)
The architectural difference means LSTMs have linear complexity with sequence length but
cannot leverage parallel processing as effectively as transformer models.
""",
        "sources": ["attention_is_all_you_need.pdf", "lstm_networks.pdf"]
    }
)

### 3.1 SummarizerAgent Evaluation

In [9]:
print("\n" + "="*60)
print("SUMMARIZER AGENT EVALUATION")
print("="*60)

async def test_summarizer():
    query = "Summarize the main points of this paper"
    response = await summarizer_agent.run(query, mock_context)
    return response

response = await test_summarizer()
print(f"\nQuery: 'Summarize the main points of this paper'")
print(f"\nResponse ({len(response.content)} chars):")
print(response.content[:500] + "..." if len(response.content) > 500 else response.content)
print(f"\nSources: {response.sources}")


SUMMARIZER AGENT EVALUATION

Query: 'Summarize the main points of this paper'

Response (1823 chars):
## Summary: Attention Is All You Need

**TL;DR:** Introduces the **Transformer**, a non-recurrent, **self-attention** architecture that achieves strong machine translation results (28.4 BLEU on WMT14 English→German) while noting scalability limits.

### Key Findings
• The **Transformer** architecture is based entirely on **self-attention**, dispensing with recurrence and convolutions.  
• **Multi-head attention** lets the model jointly attend to different representation subspaces at different po...

Sources: ['attention_is_all_you_need.pdf', 'lstm_networks.pdf']


### 3.2 MethodologyExtractorAgent Evaluation

In [10]:
print("\n" + "="*60)
print("METHODOLOGY EXTRACTOR AGENT EVALUATION")
print("="*60)

async def test_methodology():
    query = "What methodology does this paper use?"
    response = await methodology_extractor_agent.run(query, mock_context)
    return response

response = await test_methodology()
print(f"\nQuery: 'What methodology does this paper use?'")
print(f"\nResponse ({len(response.content)} chars):")
print(response.content[:500] + "..." if len(response.content) > 500 else response.content)
print(f"\nSources: {response.sources}")


METHODOLOGY EXTRACTOR AGENT EVALUATION

Query: 'What methodology does this paper use?'

Response (1761 chars):
## Methodology: Attention Is All You Need

### Research Approach
• **Type:** Experimental (proposal of a new model architecture with empirical evaluation)  
• **Design:** Proposal and empirical evaluation of a neural sequence-to-sequence architecture that completely replaces recurrence and convolution with **self-attention**; uses **multi-head attention** to allow joint attention to different representation subspaces at different positions.

### Data & Materials
• **Dataset:** **WMT 2014 English...

Sources: ['attention_is_all_you_need.pdf', 'lstm_networks.pdf']


### 3.3 ComparatorAgent Evaluation

In [11]:
print("\n" + "="*60)
print("COMPARATOR AGENT EVALUATION")
print("="*60)

async def test_comparator():
    query = "Compare the self-attention approach to recurrent models"
    response = await comparator_agent.run(query, mock_context)
    return response

response = await test_comparator()
print(f"\nQuery: 'Compare the self-attention approach to recurrent models'")
print(f"\nResponse ({len(response.content)} chars):")
print(response.content[:500] + "..." if len(response.content) > 500 else response.content)
print(f"\nSources: {response.sources}")


COMPARATOR AGENT EVALUATION

Query: 'Compare the self-attention approach to recurrent models'

Response (4173 chars):
## Comparison: attention_is_all_you_need.pdf vs lstm_networks.pdf

### Overview
• **Paper A:** attention_is_all_you_need.pdf — Introduces the **Transformer**, an architecture built entirely on **self-attention** (specifically **multi-head attention**) that dispenses with recurrence and convolutions; evaluated on machine translation (WMT 2014 En→De, En→Fr) and reports strong BLEU improvements.  
• **Paper B:** lstm_networks.pdf — Describes **Long Short-Term Memory (LSTM)** networks, a class of **...

Sources: ['attention_is_all_you_need.pdf', 'lstm_networks.pdf']


### 3.4 GapFinderAgent Evaluation

In [12]:
print("\n" + "="*60)
print("GAP FINDER AGENT EVALUATION")
print("="*60)

async def test_gap_finder():
    query = "What are the limitations of this approach?"
    response = await gap_finder_agent.run(query, mock_context)
    return response

response = await test_gap_finder()
print(f"\nQuery: 'What are the limitations of this approach?'")
print(f"\nResponse ({len(response.content)} chars):")
print(response.content[:500] + "..." if len(response.content) > 500 else response.content)
print(f"\nSources: {response.sources}")


GAP FINDER AGENT EVALUATION

Query: 'What are the limitations of this approach?'

Response (5436 chars):
## Research Gaps Analysis

### Identified Gaps

#### 1. Scalability of Self‑Attention (Computational Complexity)
• **Description:** Self‑attention in the Transformer has quadratic time and memory complexity in sequence length, making it impractical for very long sequences. Concrete efficient alternatives are not explored in the provided content.  
• **Why it matters:** Quadratic scaling limits use of Transformers on long documents, long audio/video streams, genomics, or other high‑resolution seq...

Sources: ['attention_is_all_you_need.pdf', 'lstm_networks.pdf']


### 3.5 CitationAnalyzerAgent Evaluation

In [13]:
print("\n" + "="*60)
print("CITATION ANALYZER AGENT EVALUATION")
print("="*60)

async def test_citation():
    query = "What papers does this cite or reference?"
    response = await citation_analyzer_agent.run(query, mock_context)
    return response

response = await test_citation()
print(f"\nQuery: 'What papers does this cite or reference?'")
print(f"\nResponse ({len(response.content)} chars):")
print(response.content[:500] + "..." if len(response.content) > 500 else response.content)
print(f"\nSources: {response.sources}")


CITATION ANALYZER AGENT EVALUATION

Query: 'What papers does this cite or reference?'

Response (2968 chars):
## Citation Analysis: Attention Is All You Need

### Key References
List the most important references cited:
• **(Unnamed in excerpt):** LSTM networks (lstm_networks.pdf) - The paper explicitly references LSTM-style recurrent models as the contrasting prior approach for sequence modelling; LSTMs are significant because they represent the dominant pre-Transformer architecture for handling long-term dependencies and sequential processing.  
• **(Dataset/Benchmark, unnamed in excerpt):** WMT 2014 ...

Sources: ['attention_is_all_you_need.pdf', 'lstm_networks.pdf']


### 3.6 GeneralQAAgent Evaluation

In [14]:
print("\n" + "="*60)
print("GENERAL QA AGENT EVALUATION")
print("="*60)

async def test_general_qa():
    query = "What were the main results?"
    response = await general_qa_agent.run(query, mock_context)
    return response

response = await test_general_qa()
print(f"\nQuery: 'What were the main results?'")
print(f"\nResponse ({len(response.content)} chars):")
print(response.content[:500] + "..." if len(response.content) > 500 else response.content)
print(f"\nSources: {response.sources}")


GENERAL QA AGENT EVALUATION

Query: 'What were the main results?'

Response (1468 chars):
Summary of the main results (from the provided excerpts):

- Model/architectural contribution:
  - Introduced the Transformer: an architecture based entirely on self-attention (no recurrence or convolutions) that uses multi‑head attention to “jointly attend to information from different representation subspaces at different positions.” (attention_is_all_you_need.pdf, p.3)

- Empirical performance:
  - On WMT 2014 machine translation tasks, the Transformer was evaluated on English→German and Engl...

Sources: ['attention_is_all_you_need.pdf', 'lstm_networks.pdf']


## 4. Performance Metrics Summary

In [15]:
print("\n" + "="*60)
print("AGENT EVALUATION SUMMARY")
print("="*60)

print(f"\nAgent Routing Accuracy: {accuracy:.1f}%")
print(f"\nTotal Agents Evaluated: 6")
print(f"  - SummarizerAgent: ✓")
print(f"  - MethodologyExtractorAgent: ✓")
print(f"  - ComparatorAgent: ✓")
print(f"  - GapFinderAgent: ✓")
print(f"  - CitationAnalyzerAgent: ✓")
print(f"  - GeneralQAAgent: ✓")

print(f"\nKey Findings:")
print(f"  1. All agents successfully process queries and generate responses")
print(f"  2. Agent routing via keyword matching is {accuracy:.1f}% accurate")
print(f"  3. Each agent provides specialized responses based on its role")
print(f"  4. Source citations are consistently included in responses")


AGENT EVALUATION SUMMARY

Agent Routing Accuracy: 87.5%

Total Agents Evaluated: 6
  - SummarizerAgent: ✓
  - MethodologyExtractorAgent: ✓
  - ComparatorAgent: ✓
  - GapFinderAgent: ✓
  - CitationAnalyzerAgent: ✓
  - GeneralQAAgent: ✓

Key Findings:
  1. All agents successfully process queries and generate responses
  2. Agent routing via keyword matching is 87.5% accurate
  3. Each agent provides specialized responses based on its role
  4. Source citations are consistently included in responses


## 5. Save Results

In [16]:
# Save results to JSON
results = {
    "routing_accuracy": accuracy,
    "routing_tests": routing_results,
    "agents_evaluated": [agent.name for agent in registered_agents],
    "test_context_size": len(sample_context),
    "evaluation_date": "2025-12-21"
}

results_path = Path("../results/agent_evaluation_results.json")
results_path.parent.mkdir(parents=True, exist_ok=True)

with open(results_path, 'w') as f:
    json.dump(results, f, indent=2)

print(f"\nResults saved to: {results_path}")


Results saved to: ..\results\agent_evaluation_results.json
